# Hidden states for the disagreement probe (Qwen2.5-Omni-7B, Kaggle, 2× T4)

Does the model hold more information about which clips the annotators disagree on than its output entropy
shows? This notebook reads every clip (train, valid, test) once under T+A+V with the prompt of the other
runs and saves, at the last position (the one the label is read from), the hidden state after several
layers of the language model. Each clip is read twice: with the RQ3 soft-label LoRA adapter (`A`) and without
it (`base`, the zero-shot model). The 7 label logits are saved too, to check against the earlier runs.
A linear probe is then trained on these states on a CPU (`analysis/probe_analysis.py`).

**Running it**
1. *Accelerator*: **GPU T4 x2**; *Internet*: **on**; *Add input*: the preprocessed `chsims-mm` dataset **and the
   output of the RQ3 soft-label notebook (seed 0)**, which holds `rq3_soft/best/adapter.pt`.
2. First set `LIMIT = 10` and run all interactively (about 5 minutes), then **Save & Run All (Commit)** with
   `LIMIT = None` (about 2-3 hours). If it stops at `TIME_BUDGET_H`, add this notebook's own output as an input
   and commit again: finished clips are skipped.
3. Results: `/kaggle/working/probe_hidden/`: `shard_*.npz` (uids, hidden states, label logits) and `meta.json`.
   About 350 MB; download the folder and copy it to `results/probe_hidden/`.

In [ ]:
# peft added for LoRA; the rest as in the other notebooks.
%pip install -q "transformers==5.20.1" "qwen-omni-utils==0.0.9" "peft==0.21.2" audioread librosa
# Kaggle ships torchao 0.10; peft 0.21 raises an ImportError for torchao < 0.16 even when it is not used.
%pip uninstall -y -q torchao

In [ ]:
import glob
import json
import os
import re
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

MODEL_ID = os.environ.get("MODEL_ID", "Qwen/Qwen2.5-Omni-7B")  # env override: local tests with a tiny model
LOAD_IN_4BIT = False         # True: 4-bit NF4 weights (needs bitsandbytes), if fp16 does not fit
MAX_MEMORY = {0: "12GiB", 1: "14GiB"}  # leave headroom on GPU 0 for activations
FRAME_MAX_PIXELS = 448 * 252           # per-frame pixel cap passed to the Qwen video processor
DRY_RUN = os.environ.get("DRY_RUN") == "1"  # build inputs only, no model (for local checks)
SEED = 0

# Folder made by analysis/preprocess_chsims.py; found automatically under /kaggle/input.
DATA_DIR = os.environ.get("CHSIMS_MM_DIR")
if DATA_DIR is None:
    hits = [h for k in range(6) for h in sorted(glob.glob("/kaggle/input/" + "*/" * k + "manifest.csv"))]  # no ** : media folders make it slow
    assert hits, "No manifest.csv under /kaggle/input: attach the preprocessed dataset."
    DATA_DIR = os.path.dirname(hits[0])
DATA_DIR = Path(DATA_DIR)

LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05  # as in the RQ3 notebooks
LORA_TARGETS = r"^model\.layers\.\d+\.self_attn\.(q|k|v|o)_proj$"
SHARD = 250                  # clips per saved shard
TIME_BUDGET_H = 11.0
LIMIT = None                 # e.g. 10 for a quick check: 10 clips per split
OUT_DIR = Path(os.environ.get("OUT_DIR", "/kaggle/working/probe_hidden"))
T_START = time.time()

ADAPTER = os.environ.get("ADAPTER")
if ADAPTER is None:
    hits = [h for k in range(6) for h in sorted(glob.glob("/kaggle/input/" + "*/" * k + "rq3_soft/best/adapter.pt"))]
    assert hits, "No rq3_soft/best/adapter.pt under /kaggle/input: attach the output of the RQ3 soft notebook (seed 0)."
    ADAPTER = hits[0]
print("adapter:", ADAPTER)

OUT_DIR.mkdir(parents=True, exist_ok=True)
print("data:", DATA_DIR, "| out:", OUT_DIR, "| dry run:", DRY_RUN)
print("free disk on / (GB):", round(shutil.disk_usage("/").free / 1e9, 1))
for i in range(torch.cuda.device_count()):
    print(f"cuda:{i}", torch.cuda.get_device_name(i), round(torch.cuda.get_device_properties(i).total_memory / 2**30, 1), "GiB")

## Load the model and the RQ3 soft adapter

Inference only: fp16, no gradient checkpointing. Forward hooks keep the last position of the residual stream
after each layer in `LAYERS` (a quarter, half, three quarters of the way, and the last layer) and of the final
norm (the input of the language-model head).

In [ ]:
from peft import LoraConfig, get_peft_model, set_peft_model_state_dict
from qwen_omni_utils import process_mm_info
from transformers import Qwen2_5OmniProcessor, Qwen2_5OmniThinkerForConditionalGeneration

ON_GPU = torch.cuda.is_available()
processor = Qwen2_5OmniProcessor.from_pretrained(MODEL_ID)
kwargs = dict(dtype=torch.float16 if ON_GPU else torch.float32, attn_implementation="sdpa")
if ON_GPU:
    kwargs.update(device_map="auto", max_memory=MAX_MEMORY)
t0 = time.time()
base = Qwen2_5OmniThinkerForConditionalGeneration.from_pretrained(MODEL_ID, **kwargs).eval()
print(f"loaded in {time.time() - t0:.0f} s")
base.lm_head.register_forward_pre_hook(lambda module, args: (args[0][:, -1:],) + args[1:])
model = get_peft_model(base, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                                        target_modules=LORA_TARGETS, bias="none"))
result = set_peft_model_state_dict(model, torch.load(ADAPTER, map_location="cpu"))
assert not [k for k in result.unexpected_keys if "lora" in k], result.unexpected_keys
model.eval()

layers = base.model.layers
n = len(layers)
LAYERS = sorted({max(1, n // 4), max(1, n // 2), max(1, 3 * n // 4), n})  # hidden state after layer i (1-based)
HIDDEN = {}
for i in LAYERS:
    layers[i - 1].register_forward_hook(lambda mod, args, out, i=i: HIDDEN.__setitem__(
        f"layer{i}", (out[0] if isinstance(out, tuple) else out)[0, -1].float().cpu()))
base.model.norm.register_forward_hook(lambda mod, args, out: HIDDEN.__setitem__("norm", out[0, -1].float().cpu()))
print("layers:", n, "| saved:", LAYERS, "+ norm | hidden size:", base.config.text_config.hidden_size
      if hasattr(base.config, "text_config") else base.config.hidden_size)

## Prompts (same as the other runs)

In [ ]:
LABELS = ["Anger", "Disgust", "Fear", "Joy", "Neutral", "Sadness", "Surprise"]
SYSTEM = ("You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, capable of "
          "perceiving auditory and visual inputs, as well as generating text and speech.")
DESCRIBE = {
    "TAV": "the video frames, the audio track and the transcript",
    "T": "only the transcript",
    "A": "only the audio track",
    "V": "only the video frames, without sound",
}
CONDITIONS = list(DESCRIBE)
ASK = {
    # one label; scored from the logits of its first token
    "label": f"Which emotion does the speaker express? Choose exactly one of: {', '.join(LABELS)}.\nAnswer with the label only.",
    # verbalized distribution
    "distribution": ("Imagine 100 people watched this clip. Estimate how many of them would choose each emotion "
                     f"from: {', '.join(LABELS)}. Reply with JSON only, mapping each label to a count; the counts sum to 100."),
    # RQ3: judge each modality on its own, then aggregate
    "per_modality": (
        "Judge each modality on its own before deciding:\n"
        "- text: what the words alone suggest;\n"
        "- audio: what the tone of voice alone suggests;\n"
        "- video: what the face and body alone suggest.\n"
        "Then say whether the modalities conflict, and estimate how 100 people who watched the whole clip "
        "would split across the emotions.\n"
        f"Use only these labels: {', '.join(LABELS)}. Reply with JSON only, in this form:\n"
        '{"text": {"cue": "<a few words>", "label": "<label>"}, "audio": {"cue": "...", "label": "..."}, '
        '"video": {"cue": "...", "label": "..."}, "conflict": true or false, "distribution": {"<label>": <count>, ...}}'),
}

label_first_ids = [processor.tokenizer.encode(l, add_special_tokens=False)[0] for l in LABELS]
assert len(set(label_first_ids)) == len(LABELS), "two labels share a first token"
# The checkpoint's generation_config sets no end-of-turn token, so without this the model keeps
# writing ("Human: ...") until max_new_tokens.
STOP_IDS = processor.tokenizer.convert_tokens_to_ids(["<|im_end|>", "<|endoftext|>"])


def build_messages(row, condition, ask="label"):
    content = []
    if "V" in condition:
        frames = sorted(glob.glob(str(DATA_DIR / row["frames_dir"] / "*.jpg")))
        content.append({"type": "video", "video": [f"file://{f}" for f in frames],
                        "sample_fps": float(row["sample_fps"]), "max_pixels": FRAME_MAX_PIXELS})
    if "A" in condition:
        content.append({"type": "audio", "audio": str(DATA_DIR / row["audio_path"])})
    text = f"You are given {DESCRIBE[condition]} of a speaker in a scene from a Chinese film or TV series.\n"
    if "T" in condition:
        text += f"Transcript: {row['text']}\n"
    content.append({"type": "text", "text": text + ASK[ask]})
    return [{"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
            {"role": "user", "content": content}]


def prepare(messages):
    text = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    audios, images, videos, video_kwargs = process_mm_info(messages, use_audio_in_video=False, return_video_kwargs=True)
    # qwen-omni-utils returns fps as a per-video list; the transformers 5.x processor wants one number.
    fps = video_kwargs.pop("fps", [])
    if fps:
        video_kwargs["fps"] = float(fps[0])
    inputs = processor(text=text, audio=audios, images=images, videos=videos, return_tensors="pt",
                       padding=True, use_audio_in_video=False, **video_kwargs)
    if model is not None:
        inputs = {k: (v.to(model.device, dtype=model.dtype) if torch.is_floating_point(v) else v.to(model.device))
                  if torch.is_tensor(v) else v for k, v in inputs.items()}
    return inputs


def parse_json(text):
    match = re.search(r"\{.*\}", text, flags=re.S)
    try:
        return json.loads(match.group(0)) if match else None
    except (json.JSONDecodeError, ValueError):
        return None

## Clips

In [ ]:
manifest = pd.read_csv(DATA_DIR / "manifest.csv", dtype={"video_id": str, "clip_id": str})
clips = manifest[manifest["status"] == "ok"].reset_index(drop=True)
order = {"test": 0, "valid": 1, "train": 2}  # test first, so a short run still covers the evaluation split
clips = clips.sort_values("mode", key=lambda m: m.map(order), kind="stable").reset_index(drop=True)
if LIMIT:
    clips = clips.groupby("mode", sort=False).head(LIMIT).reset_index(drop=True)
print(clips["mode"].value_counts().to_dict())

## Read every clip twice and save shards

Per clip and model (`A`, `base`): the hidden states and the 7 label logits. Shards found under `/kaggle/input`
(an earlier run of this notebook) or in `OUT_DIR` are kept and their clips skipped.

In [ ]:
for p in (h for k in range(6) for h in glob.glob("/kaggle/input/" + "*/" * k + "probe_hidden/shard_*.npz")):
    if not (OUT_DIR / Path(p).name).exists():
        shutil.copy(p, OUT_DIR / Path(p).name)
done = set()
for p in sorted(OUT_DIR.glob("shard_*.npz")):
    done |= set(np.load(p, allow_pickle=False)["uid"].tolist())
todo = clips[~clips["uid"].isin(done)]
print(f"{len(clips)} clips, {len(done & set(clips['uid']))} already done, {len(todo)} to run")


@torch.inference_mode()
def read(inputs):
    HIDDEN.clear()
    logits = model(**inputs, use_cache=False).logits[0, -1].float()
    return {k: v.numpy().astype(np.float16) for k, v in HIDDEN.items()}, logits[label_first_ids].cpu().numpy()


buf, n_oom, t_run = [], 0, time.time()


def flush():
    if not buf:
        return
    k = len(list(OUT_DIR.glob("shard_*.npz")))
    arrays = {"uid": np.array([b["uid"] for b in buf])}
    for key in buf[0]:
        if key != "uid":
            arrays[key] = np.stack([b[key] for b in buf])
    np.savez(OUT_DIR / f"shard_{k:03d}.npz", **arrays)
    buf.clear()


for j, (_, row) in enumerate(todo.iterrows()):
    if (time.time() - T_START) / 3600 > TIME_BUDGET_H:
        print("stopped at the time budget: add this notebook's output as an input and commit again")
        break
    try:
        inputs = prepare(build_messages(row, "TAV", ask="label"))
        rec = {"uid": row["uid"]}
        for name in ("A", "base"):
            if name == "A":
                hidden, logits = read(inputs)
            else:
                with model.disable_adapter():
                    hidden, logits = read(inputs)
            rec.update({f"{name}_{k}": v for k, v in hidden.items()})
            rec[f"{name}_logits"] = logits
        buf.append(rec)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        n_oom += 1
    if len(buf) == SHARD:
        flush()
    if j + 1 in (5, 50) or (j + 1) % 500 == 0:
        sec = (time.time() - t_run) / (j + 1)
        peak = max([torch.cuda.max_memory_allocated(d) / 2**30 for d in range(torch.cuda.device_count())] or [0])
        print(f"{j + 1}/{len(todo)} | {sec:.2f} s/clip, about {(len(todo) - j - 1) * sec / 3600:.1f} h left | peak {peak:.1f} GiB | OOM {n_oom}")
flush()
(OUT_DIR / "meta.json").write_text(json.dumps({"model": MODEL_ID, "adapter": ADAPTER, "layers": LAYERS, "labels": LABELS,
                                               "oom_this_run": n_oom}, indent=1))

## Check

In [ ]:
total = {}
for p in sorted(OUT_DIR.glob("shard_*.npz")):
    z = np.load(p, allow_pickle=False)
    for u in z["uid"].tolist():
        total[u] = True
    keys = [k for k in z.files if k != "uid"]
print(f"{len(total)} of {len(clips)} clips saved | arrays per clip:", {k: z[k].shape[1:] for k in keys})
print(clips[clips["uid"].isin(total)]["mode"].value_counts().to_dict())